# 6장 시각화와 기술통계

이 노트북은 웹 교재 [6장 시각화와 기술통계](https://pidocs.aifusionlab.org/ai-python/ch06/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/seoul_weather_2021_2025.csv
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 6.1 matplotlib 그래프

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch06/6-1/)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("seoul_weather_2021_2025.csv", parse_dates=["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
monthly = df.groupby("month").agg(temp=("temp_mean", "mean"), precip=("precip_mm", "sum"))
monthly["precip"] = monthly["precip"] / 5

In [ ]:
y25 = df[df["year"] == 2025]
plt.figure(figsize=(10, 3.5))
plt.plot(y25["date"], y25["temp_mean"], label="mean")
plt.fill_between(y25["date"], y25["temp_min"], y25["temp_max"], alpha=0.25, label="min-max")
plt.ylabel("temperature (C)")
plt.title("Seoul 2025 daily temperature")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.bar(monthly.index, monthly["precip"])
plt.xticks(range(1, 13))
plt.xlabel("month")
plt.ylabel("precipitation (mm)")
plt.title("Average monthly precipitation 2021-2025")
plt.show()

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.hist(df["temp_mean"], bins=30)
plt.xlabel("daily mean temperature (C)")
plt.ylabel("days")
plt.show()

## 6.2 기술통계와 분포

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch06/6-2/)

In [ ]:
print(df["temp_mean"].agg(["mean", "median", "std"]).round(2))
print(df["precip_mm"].agg(["mean", "median", "std"]).round(2))
df["precip_mm"].quantile([0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
rain = df[df["precip_mm"] >= 1.0]["precip_mm"]
print(len(rain), rain.median(), rain.quantile(0.75))
plt.figure(figsize=(7, 2.8))
plt.boxplot(rain, vert=False)
plt.xlabel("daily precipitation on rain days (mm)")
plt.show()

## 6.3 실습: 계절별 기온 분포와 센서 분포

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch06/6-3/)

In [ ]:
season_of = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
             6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
df["season"] = df["month"].map(season_of)
order = ["DJF", "MAM", "JJA", "SON"]

plt.figure(figsize=(6, 3.5))
plt.boxplot([df[df["season"] == s]["temp_mean"] for s in order], tick_labels=order)
plt.ylabel("daily mean temperature (C)")
plt.show()
df.groupby("season")["temp_mean"].describe().loc[order].round(1)

In [ ]:
rssi = pd.read_csv("class_rssi_20261104.csv")
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, z in zip(axes, ["D1", "D2", "D3", "D4"]):
    ax.hist(rssi[rssi["zone"] == z]["rssi"], bins=range(-90, -30, 3))
    ax.set_title(z)
    ax.set_xlabel("RSSI (dBm)")
axes[0].set_ylabel("records")
plt.tight_layout()
plt.show()
rssi.groupby("zone")["rssi"].quantile([0.1, 0.5, 0.9]).unstack().round(1)